# 06 · Scaling out: several GPUs, several vLLM servers

Phase 5 tuned one RTX 5090. This phase asks what happens with **several GPUs in one machine**: how well throughput scales with more vLLM servers, how NGINX should spread the load, whether splitting one model across GPUs beats running copies, what happens when a server dies under load, and what Mooncake adds once there's more than one server.

Every vLLM server runs the Phase 5 recommended config (**FP8 weights**). All numbers are read from `results/06_scaling_out/`.

## Setups

`scripts/topology.py` writes the Compose, NGINX and Prometheus config for each setup:

| Experiment | Setup | Traffic |
|---|---|---|
| Scaling | 1, 2, 4 copies (one per GPU), round robin | Phase 4 mix; users scaled with the number of copies |
| Load balancing | 4 copies × round robin / least connections / sticky per user | **Conversations**: each user discusses a long document for 6 turns, so its prefix grows and repeats |
| Failover | 4 copies, least connections; one copy stopped under load for 90 s | Phase 4 mix |
| Tensor parallel | 1 server split across 2 (and 4) GPUs | Same as the 2 (4) copy scaling run |
| Mooncake store | 4 copies, round robin, one KV store shared through Mooncake | Conversations |
| Disaggregated | 1 prefill + 1 decode server, KV handed over by Mooncake | Same as the 2 copy run |

The gateway runs 8 worker processes here: in Phase 4 one process used 84% of a core at ~6,000 tokens/s, and four GPUs produce several times that.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = ROOT / "results" / "06_scaling_out"
sys.path.insert(0, str(ROOT / "loadtest"))
from results_logger import load_runs, summarize

runs = load_runs("06_scaling_out")
sweeps = {r["config"]["label"]: r for r in runs if r["name"].startswith("load_sweep_")}
evals = {r["config"]["label"]: r for r in runs if r["name"].startswith("evals_")}
timeline = next((r for r in reversed(runs) if r["name"].startswith("timeline_")), None)
print(f"sweeps: {sorted(sweeps)}\nevals: {sorted(evals)} | failover timeline: {bool(timeline)}")
for f in sorted(RESULTS.glob("failed_*.log")):
    print(f"DID NOT START: {f.name}")
if sweeps:
    env = next(iter(sweeps.values()))["environment"]
    print(f"GPUs: {len(env['gpus'])} × {env['gpus'][0]['name'] if env['gpus'] else '?'}")

def v(d, *keys):
    for k in keys:
        d = (d or {}).get(k)
    return d

def cell(value, fmt, width, scale=1, unit=""):
    return ("-" if value is None else format(value * scale, fmt) + unit).rjust(width)

def sweep_table(label):
    print(f"{label}  ({sweeps[label]['config'].get('topology')})")
    print(f"{'users':>7}{'tok/s':>8}{'TTFT p95':>11}{'ITL p50':>10}{'running':>9}{'waiting':>9}{'KV max':>8}{'errors':>8}")
    for s in sweeps[label]["metrics"]["steps"]:
        sv = s.get("server", {})
        print(f"{s['users']:>7}{cell(sv.get('output_tokens_per_s'), '.0f', 8)}{cell(s['ttft_s']['p95'], '.0f', 11, 1000, 'ms')}"
              f"{cell(s['itl_s']['p50'], '.1f', 10, 1000, 'ms')}{cell(v(sv, 'running', 'avg'), '.0f', 9)}"
              f"{cell(v(sv, 'waiting', 'max'), '.0f', 9)}{cell(v(sv, 'kv_cache_usage', 'max'), '.0%', 8)}{s['errors']:>8}")
    print()

## 1. Scaling: 1 → 2 → 4 copies

In [ ]:
scale = [l for l in ("scale_1x", "scale_2x", "scale_4x") if l in sweeps]
for l in scale:
    sweep_table(l)

In [ ]:
def peak(label):
    steps = sweeps[label]["metrics"]["steps"]
    best = max(steps, key=lambda s: v(s, "server", "output_tokens_per_s") or 0)
    return v(best, "server", "output_tokens_per_s"), best["users"]

if scale:
    base = peak("scale_1x")[0] if "scale_1x" in sweeps else None
    print(f"{'setup':<10}{'GPUs':>5}{'peak tok/s':>12}{'per GPU':>10}{'speedup':>9}{'efficiency':>12}")
    for l in scale:
        n = int(l.split("_")[1].rstrip("x"))
        p, users = peak(l)
        print(f"{l:<10}{n:>5}{p:>12.0f}{p / n:>10.0f}{(p / base if base else 0):>8.2f}×{(p / base / n if base else 0):>11.0%}   (at {users} users)")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for l in scale:
        steps = sweeps[l]["metrics"]["steps"]
        axes[0].plot([s["users"] for s in steps], [v(s, "server", "output_tokens_per_s") for s in steps], "o-", label=l)
        axes[1].plot([s["users"] for s in steps], [s["ttft_s"]["p95"] for s in steps], "o-", label=l)
    axes[0].set_title("Output tokens/s"); axes[1].set_title("TTFT p95 (s, log)"); axes[1].set_yscale("log")
    for ax in axes:
        ax.set_xscale("log", base=2); ax.set_xlabel("concurrent users"); ax.legend()
    plt.tight_layout(); plt.show()

## 2. Load balancing with conversations

Each user holds a 6-turn conversation about a long document, so every request repeats that user's growing prefix. Sticky routing (`hash $http_x_user_id`) keeps a user on one server, where the prefix is already cached; round robin and least connections spread a user's turns over all four.

In [ ]:
lbs = [l for l in ("lb_round_robin", "lb_least_conn", "lb_user_hash", "lb_round_robin_mooncake") if l in sweeps]
if lbs:
    print(f"{'setup':<26}{'users':>6}{'tok/s':>8}{'TTFT p50':>10}{'TTFT p95':>10}{'prefix hit':>12}{'Mooncake hit':>14}   running per server")
    for l in lbs:
        for s in sweeps[l]["metrics"]["steps"]:
            sv = s.get("server", {})
            per = sv.get("running_per_replica") or {}
            print(f"{l:<26}{s['users']:>6}{cell(sv.get('output_tokens_per_s'), '.0f', 8)}{cell(s['ttft_s']['p50'], '.0f', 10, 1000, 'ms')}"
                  f"{cell(s['ttft_s']['p95'], '.0f', 10, 1000, 'ms')}{cell(sv.get('prefix_cache_hit_rate'), '.0%', 12)}"
                  f"{cell(sv.get('external_cache_hit_rate'), '.0%', 14)}   " + " ".join(f"{per[k]:.0f}" for k in sorted(per)))
    print("\nTTFT p95 by request type (opening a conversation vs a follow-up turn), last step:")
    for l in lbs:
        last = sweeps[l]["metrics"]["steps"][-1]
        print(f"  {l:<26}" + "   ".join(f"{c}: {cell(x.get('ttft_p95_s'), '.0f', 0, 1000, 'ms')}" for c, x in sorted(last["by_category"].items())))

## 3. Failover: one server stopped under load

In [ ]:
if timeline:
    m = timeline["metrics"]
    tl = m["timeline"]
    print(f"{m['requests']} requests, {m['errors']} errors")
    for e in m["events"]:
        print(f"  {e['t_s']:.0f} s: {e['event']}")
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.bar([b["t_s"] for b in tl], [b["ok"] for b in tl], width=8, label="finished OK")
    ax.bar([b["t_s"] for b in tl], [b["errors"] for b in tl], width=8, bottom=[b["ok"] for b in tl], color="#d94801", label="errors")
    for e in m["events"]:
        ax.axvline(e["t_s"], color="black", ls="--", lw=1); ax.text(e["t_s"] + 2, ax.get_ylim()[1] * 0.9, e["event"])
    ax2 = ax.twinx(); ax2.plot([b["t_s"] for b in tl], [b["ttft_p95_s"] for b in tl], color="#6a51a3", label="TTFT p95")
    ax2.set_ylabel("TTFT p95 (s)"); ax.set_xlabel("seconds"); ax.set_ylabel("requests per 10 s"); ax.legend(loc="upper left")
    plt.tight_layout(); plt.show()
    for b in tl:
        if b["errors"]:
            print(f"  {b['t_s']:.0f} s: {b['errors']} errors, e.g. {b['error_samples'][:2]}")
else:
    print("No failover run.")

## 4. Tensor parallel vs copies

In [ ]:
pairs = [(a, b) for a, b in (("scale_2x", "tp2"), ("scale_4x", "tp4")) if a in sweeps and b in sweeps]
for a, b in pairs:
    print(f"{'users':>7}{a + ' tok/s':>16}{b + ' tok/s':>14}{a + ' TTFT p95':>18}{b + ' TTFT p95':>16}{a + ' ITL p50':>17}{b + ' ITL p50':>15}")
    sb = {s["users"]: s for s in sweeps[b]["metrics"]["steps"]}
    for s in sweeps[a]["metrics"]["steps"]:
        t = sb.get(s["users"], {})
        print(f"{s['users']:>7}{cell(v(s, 'server', 'output_tokens_per_s'), '.0f', 16)}{cell(v(t, 'server', 'output_tokens_per_s'), '.0f', 14)}"
              f"{cell(s['ttft_s']['p95'], '.0f', 18, 1000, 'ms')}{cell(v(t, 'ttft_s', 'p95'), '.0f', 16, 1000, 'ms')}"
              f"{cell(s['itl_s']['p50'], '.1f', 17, 1000, 'ms')}{cell(v(t, 'itl_s', 'p50'), '.1f', 15, 1000, 'ms')}")
    print()
if not pairs:
    print("No tensor parallel runs.")

## 5. Disaggregated prefill and decode

In [ ]:
if "pd_1p1d" in sweeps and "scale_2x" in sweeps:
    sweep_table("pd_1p1d")
    sweep_table("scale_2x")
else:
    print("No disaggregated run.")

## Answer quality on the new setups

The 22-case eval set from Phase 3 on the setups that change how the model computes (tensor parallel splits every matrix multiply across GPUs; disaggregated serving moves the KV cache between servers). Phase 5's single-GPU FP8 weights result was 21 of 22.

In [ ]:
for l, r in sorted(evals.items()):
    s = r["metrics"]["summary"]
    print(f"{l:<22}{s['passed']}/{s['total']}   " + ", ".join(f"{k} {x['passed']}/{x['total']}" for k, x in s["by_category"].items()))
    for c in r["metrics"]["cases"]:
        if not c["passed"]:
            print(f"     FAIL {c['id']:<22} {c['result']['content'][:80]!r}")

## Grafana

*(added after the GPU session)*

## What happened

*(written after the GPU session, from the numbers above)*

## Phase 6 exit criteria

- [x] Generated setups for N copies, tensor parallel, load balancing methods, a shared Mooncake store and disaggregated serving
- [x] Conversation load mode, per-server balance and prefix cache hit rate per step
- [x] Failover drill under load with a per-10-second timeline
- [ ] Phase 6 session on a multi-GPU RTX 5090 machine
- [ ] Scaling factor, best routing, failover behaviour and Mooncake results explained